# 线性探针

Happy MI：动手学机制可解释性

合成数据中，下游只使用第一个坐标，探针读取第二个坐标。先预测探针准确率与干预效应是否一致。

正文与图示 CC BY-NC-SA 4.0；代码 MIT。实现由 experiments/probes.py 自动生成。

## 环境

以下单元会联网安装固定教学依赖。如果运行时提示重启，请重启后从下一节开始。推荐 Python 3.11。

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "numpy==2.3.3", "torch==2.8.0"])

In [ ]:
import random
import numpy as np
import torch

def seed_all(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.set_num_threads(1)


## 看懂实现

先读函数，标出输入、被观察或替换的变量、评价指标。此单元只定义函数。

In [ ]:
"""Train a probe for a variable explicitly unused by the target model."""

import json
import torch


def fit_probe(h, labels, steps=200):
    probe = torch.nn.Linear(h.shape[1], 1)
    optimizer = torch.optim.Adam(probe.parameters(), lr=0.05)
    for _ in range(steps):
        logits = probe(h).squeeze(-1)
        loss = torch.nn.functional.binary_cross_entropy_with_logits(logits, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    return probe


@torch.no_grad()
def accuracy(probe, h, labels):
    return ((probe(h).squeeze(-1) > 0) == labels.bool()).float().mean().item()


def run(seed=42):
    seed_all(seed)
    train = torch.randn(2048, 3)
    test = torch.randn(2048, 3)
    y_train, y_test = (train[:, 1] > 0).float(), (test[:, 1] > 0).float()
    probe = fit_probe(train, y_train)
    shuffled_train = y_train[torch.randperm(len(y_train))]
    shuffled_test = y_test[torch.randperm(len(y_test))]
    control = fit_probe(train, shuffled_train)
    modified = test.clone()
    modified[:, 1] = 0
    # The target uses coordinate 0 only; the probe reads coordinate 1.
    target_before, target_after = test[:, 0] > 0, modified[:, 0] > 0
    return {
        'kind': 'synthetic_known_mechanism', 'seed': seed,
        'train_samples': len(train), 'test_samples': len(test),
        'probe_train_accuracy': accuracy(probe, train, y_train),
        'probe_test_accuracy': accuracy(probe, test, y_test),
        'random_label_train_accuracy': accuracy(control, train, shuffled_train),
        'random_label_test_accuracy': accuracy(control, test, shuffled_test),
        'majority_test_accuracy': max(y_test.mean().item(), 1 - y_test.mean().item()),
        'target_predictions_changed': (target_before != target_after).sum().item(),
        'test_samples_with_coordinate_changed': (test[:, 1] != modified[:, 1]).sum().item(),
    }



## 运行并检查

保留原始结果，不只保留符合预期的部分。

In [ ]:
result = run()
print(json.dumps(result, ensure_ascii=False, indent=2))

## 练习

运行 run(seed=7)，比较随机标签控制与正常标签的测试准确率。

写下：哪些结果支持你的预测？哪些不支持？这个实验不能说明什么？